# 第6部 カメラ認識

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asg1223/rcj-soccer-ml/blob/main/notebooks/06_camera.ipynb)

このノートブックは、教材のページのコードを順番に並べたものです（`tools/build_notebooks.py` で自動生成）。
上のセルから順に `Shift + Enter` で実行してください。

In [ ]:
# Colab のときだけ、教材リポジトリを取ってきて読み込めるようにする
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q --depth 1 https://github.com/Asg1223/rcj-soccer-ml.git /content/rcj-soccer-ml
    sys.path.insert(0, "/content/rcj-soccer-ml")
print("Colab で実行中" if IN_COLAB else "ローカルで実行中")

## 6-1 画像はただの数字の表

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part6-camera/pixels/

### 手を動かす 1：画像を作って、数字として見る

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from rcjml.sim.camera import make_images, CLASSES

X, labels, infos = make_images(8, seed=1)
print(X.shape, X.dtype)          # (8, 96, 96, 3) uint8：8 枚、縦 96、横 96、RGB の 3 つ

COLORS = ["orange", "gold", "blue", "red"]
def show(images, box_lists, ncols=4):
    nrows = int(np.ceil(len(images) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows), squeeze=False)
    for ax, img, boxes in zip(axes.flat, images, box_lists):
        ax.imshow(img)
        for b in boxes:
            ax.add_patch(patches.Rectangle((b.x0, b.y0), b.x1 - b.x0, b.y1 - b.y0,
                                           fill=False, color=COLORS[b.cls], lw=1.5))
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show(X, labels)

### 手を動かす 2：ボールの画素の数字を見る

In [ ]:
img, boxes = X[0], labels[0]
balls = [b for b in boxes if b.cls == 0]
if balls:
    b = balls[0]
    x, y = int(b.cx), int(b.cy)
    print("ボールの真ん中の画素 (R, G, B):", img[y, x])
print("床の画素 (R, G, B):", img[90, 10])

### 手を動かす 3：HSV にして、明るさを変えても H が変わらないことを確かめる

In [ ]:
import cv2

bright = X[0]
dark = (X[0] * 0.5).astype(np.uint8)          # 照明が半分になったつもり
for name, im in [("bright", bright), ("dark", dark)]:
    hsv = cv2.cvtColor(im, cv2.COLOR_RGB2HSV)
    if balls:
        print(name, "RGB:", im[y, x], " HSV:", hsv[y, x])

### 手を動かす 4：解像度で計算量がどう変わるか

In [ ]:
for w, h in [(640, 480), (320, 240), (160, 120), (96, 96)]:
    print(f"{w}×{h}: 画素 {w * h:7d} 個、数字 {w * h * 3:8d} 個、640×480 の {w * h / (640 * 480):.3f} 倍")

## 6-2 まずは色で探す：HSV しきい値（ベースライン）

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part6-camera/hsv-baseline/

### 手を動かす 1：1 枚で試す

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from rcjml.sim.camera import make_images, HORIZON, CLASSES
from rcjml.vision import hsv_detect, evaluate, HSV_RANGES

X, labels, _ = make_images(8, seed=1)
img = X[0]
hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
lo, hi = HSV_RANGES[0][0]                       # ボール（オレンジ）の範囲
mask = cv2.inRange(hsv, np.array(lo, np.uint8), np.array(hi, np.uint8))

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img); axes[0].set_title("image")
axes[1].imshow(mask, cmap="gray"); axes[1].set_title("orange mask")
for d in hsv_detect(img):
    axes[0].plot(d.cx, d.cy, "x", color="white", ms=10, mew=2)
plt.show()
print(HSV_RANGES)

### 手を動かす 2：1,000 枚で成績を測る

In [ ]:
X, labels, _ = make_images(1000, seed=100)

def show_result(name, preds):
    r = evaluate(preds, labels)
    print(name)
    for cls, v in r.items():
        print(f"  {cls:12s} 見つけた {v['recall']:.0%}  当たった {v['precision']:.0%}  （正解 {v['n_true']} 個）")

show_result("色だけ", [hsv_detect(x) for x in X])

### 手を動かす 3：探す場所をしぼる

In [ ]:
show_result("色 ＋ 探す場所をしぼる", [hsv_detect(x, horizon=HORIZON) for x in X])

## 6-3 画像を集める

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part6-camera/collect-images/

### 手を動かす 2：撮り方の計画を立てる

In [ ]:
import numpy as np
from collections import Counter
from rcjml.sim.camera import make_images, CLASSES

X, labels, infos = make_images(1000, seed=1)
count = Counter(CLASSES[b.cls] for boxes in labels for b in boxes)
print("写っている数:", dict(count))
print("ボールが写っていない画像:", sum(not any(b.cls == 0 for b in boxes) for boxes in labels), "枚")

dist = np.array([i["ball"]["distance_cm"] for i in infos if i["ball"]])
print("ボールの距離（cm）の分布:", np.histogram(dist, bins=[0, 20, 40, 80, 160])[0])

## 6-4 ラベルを付ける

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part6-camera/labeling/

### 手を動かす 2：ノートブックで読み込む

In [ ]:
import tempfile, pathlib, cv2
from rcjml.sim.camera import make_images, boxes_to_yolo
from rcjml.vision import load_yolo_folder

X, labels, _ = make_images(20, seed=5)
root = pathlib.Path(tempfile.mkdtemp())
for k, (img, boxes) in enumerate(zip(X, labels)):
    session = root / ("day1" if k < 10 else "day2")
    session.mkdir(exist_ok=True)
    cv2.imwrite(str(session / f"img_{k:05d}.png"), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    (session / f"img_{k:05d}.txt").write_text(boxes_to_yolo(boxes))

X2, labels2, sessions2 = load_yolo_folder(root)
print(X2.shape, sorted(set(sessions2)))
print(open(next(root.rglob("*.txt"))).read())

### 手を動かす 3：撮影した日・場所で分ける

In [ ]:
import numpy as np
sessions2 = np.array(sessions2)
test_mask = sessions2 == "day2"              # 最後の撮影をテスト用に
print("学習用", (~test_mask).sum(), "枚、テスト用", test_mask.sum(), "枚")

## 6-5 CNN：画像のためのニューラルネット

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part6-camera/cnn/

### 手を動かす 1：データを用意する

In [ ]:
import numpy as np
import keras
from keras import layers
from rcjml.sim.camera import make_images, ball_position_class, HORIZON
from rcjml.vision import hsv_detect

X_train, L_train, _ = make_images(8000, seed=1)
X_val, L_val, _ = make_images(1000, seed=2)
X_test, L_test, _ = make_images(1000, seed=3)
to_class = lambda L: np.array([ball_position_class(b) for b in L])
y_train, y_val, y_test = to_class(L_train), to_class(L_val), to_class(L_test)
print("答えの数:", np.bincount(y_train))     # なし・左・真ん中・右

### 手を動かす 2：比べる相手を用意する

In [ ]:
majority = (y_test == np.bincount(y_train).argmax()).mean()
print(f"いつもいちばん多い答えを言う: {majority:.0%}")

def hsv_class(img):
    balls = [d for d in hsv_detect(img, horizon=HORIZON) if d.cls == 0]
    if not balls:
        return 0
    x = balls[0].cx
    return 1 if x < 32 else 2 if x < 64 else 3

print(f"色のしきい値（6-2）: {np.mean([hsv_class(x) == y for x, y in zip(X_test, y_test)]):.0%}")

### 手を動かす 3：CNN を作って学習する

In [ ]:
keras.utils.set_random_seed(0)
cnn = keras.Sequential([
    keras.Input((96, 96, 3)),
    layers.Rescaling(1 / 255),                                   # 0〜255 → 0〜1
    layers.Conv2D(16, 3, padding="same", activation="relu"), layers.MaxPooling2D(),   # 48×48
    layers.Conv2D(32, 3, padding="same", activation="relu"), layers.MaxPooling2D(),   # 24×24
    layers.Conv2D(64, 3, padding="same", activation="relu"), layers.MaxPooling2D(),   # 12×12
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.GlobalMaxPooling2D(),                                 # どこかにあれば大きな値になる
    layers.Dense(32, activation="relu"),
    layers.Dense(4, activation="softmax"),                       # 4 つの確率
])
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=40, batch_size=64, verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])
print(f"CNN: {cnn.evaluate(X_test, y_test, verbose=0)[1]:.0%}")

### 手を動かす 4：まちがえた画像を見る

In [ ]:
import matplotlib.pyplot as plt
pred = cnn.predict(X_test, verbose=0).argmax(1)
wrong = np.where(pred != y_test)[0][:8]
names = ["none", "left", "center", "right"]
fig, axes = plt.subplots(1, len(wrong), figsize=(2.5 * len(wrong), 2.8))
for ax, i in zip(np.atleast_1d(axes), wrong):
    ax.imshow(X_test[i])
    ax.set_title(f"true {names[y_test[i]]}\npred {names[pred[i]]}", fontsize=8)
    ax.axis("off")
plt.show()

## 6-6 物体検出：ボール・ゴール・ロボットを同時に見つける

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part6-camera/detection/

### 手を動かす 1：データを作る

In [ ]:
import numpy as np
import keras
import matplotlib.pyplot as plt
from rcjml.sim.camera import make_images, CLASSES, centroid_grid
from rcjml.vision import build_fomo, fomo_loss, fomo_detections, grid_targets, evaluate, hsv_detect

X_train, L_train, _ = make_images(8000, seed=1)
X_val, L_val, _ = make_images(1000, seed=2)
X_test, L_test, _ = make_images(1000, seed=3)

print(centroid_grid(L_train[0]))        # 1 枚目の正解のマス（0 = 何もない、1〜4 = 種類）

### 手を動かす 2：学習する

In [ ]:
keras.utils.set_random_seed(0)
fomo = build_fomo()
fomo.summary()
fomo.compile(optimizer=keras.optimizers.Adam(2e-3), loss=fomo_loss(object_weight=20.0))
fomo.fit(X_train, grid_targets(L_train),
         validation_data=(X_val, grid_targets(L_val)),
         epochs=40, batch_size=64, verbose=0,
         callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])

### 手を動かす 3：成績を測る

In [ ]:
P = fomo.predict(X_test, verbose=0)
preds = [fomo_detections(p, threshold=0.5) for p in P]
r = evaluate(preds, L_test)
for cls, v in r.items():
    print(f"{cls:12s} 見つけた {v['recall']:.0%}  当たった {v['precision']:.0%}")

In [ ]:
COLORS = ["orange", "gold", "blue", "red"]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, img, ps in zip(axes.flat, X_test[:8], preds[:8]):
    ax.imshow(img)
    for p in ps:
        ax.plot(p.cx, p.cy, "o", mfc="none", mec=COLORS[p.cls], ms=12, mew=2)
    ax.axis("off")
plt.tight_layout()
plt.show()

### 手を動かす 4：マイコン用に書き出す

In [ ]:
import tensorflow as tf
from rcjml.config import PACKAGE_ROOT
from rcjml.export import tflite_to_c

def representative_images():
    for i in range(0, 400, 4):
        yield [X_train[i:i + 1].astype(np.float32)]

conv = tf.lite.TFLiteConverter.from_keras_model(fomo)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = representative_images
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
fomo_int8 = conv.convert()
print("int8 のモデル:", len(fomo_int8), "バイト")

path = tflite_to_c(fomo_int8, PACKAGE_ROOT / "firmware" / "cores3" / "camera_detect" / "fomo_tflite.h",
                   name="fomo_tflite")

## 6-7 CoreS3 で速く動かす

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part6-camera/speed/

### 手を動かす 1：計算量を数える

In [ ]:
import keras
import pandas as pd
from rcjml.vision import build_fomo

def count_macs(model):
    macs = 0
    for layer in model.layers:
        if isinstance(layer, keras.layers.Conv2D):
            kh, kw, cin, cout = layer.kernel.shape
            oh, ow = layer.output.shape[1:3]
            macs += oh * ow * kh * kw * cin * cout
    return macs

rows = []
for size in [64, 96, 128]:
    for width in [8, 16, 32]:
        m = build_fomo(width=width, input_shape=(size, size, 3))
        rows.append({"input": f"{size}x{size}", "width": width,
                     "params": m.count_params(), "MACs (millions)": round(count_macs(m) / 1e6, 1),
                     "int8 size (KB, approx)": round(m.count_params() / 1024, 1)})
pd.DataFrame(rows)

### 手を動かす 2：幅を変えて、成績と比べる

In [ ]:
keras.utils.set_random_seed(0)
small = build_fomo(width=8)
small.compile(optimizer=keras.optimizers.Adam(2e-3), loss=fomo_loss())
small.fit(X_train, grid_targets(L_train), validation_data=(X_val, grid_targets(L_val)),
          epochs=40, batch_size=64, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
r_small = evaluate([fomo_detections(p, 0.5) for p in small.predict(X_test, verbose=0)], L_test)
for cls in CLASSES:
    print(f"{cls:12s} 幅16: 見つけた {r[cls]['recall']:.0%}   幅8: 見つけた {r_small[cls]['recall']:.0%}")
print("計算量:", count_macs(fomo) // 10**6, "→", count_macs(small) // 10**6, "百万回")

## 6-8 相手ロボットを見つける

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part6-camera/robots/

### 手を動かす 1：評価用の「知らないチーム」の画像

In [ ]:
import numpy as np
import keras
from rcjml.sim.camera import make_images
from rcjml.vision import build_fomo, fomo_loss, fomo_detections, grid_targets, evaluate, augment_images

X_val, L_val, _ = make_images(1000, seed=2)
X_unseen, L_unseen, _ = make_images(1000, seed=9, robot_styles=tuple(range(20, 30)))   # 知らない 10 チーム

### 手を動かす 2：集め方を変えて比べる

In [ ]:
def train_eval(styles, aug=False, seed=11):
    X, L, _ = make_images(6000, seed=seed, robot_styles=styles)
    if aug:
        X = augment_images(X, np.random.default_rng(0))
    keras.utils.set_random_seed(0)
    m = build_fomo()
    m.compile(optimizer=keras.optimizers.Adam(2e-3), loss=fomo_loss())
    m.fit(X, grid_targets(L), validation_data=(X_val, grid_targets(L_val)),
          epochs=25, batch_size=64, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
    r = evaluate([fomo_detections(p, 0.5) for p in m.predict(X_unseen, verbose=0)], L_unseen)
    return r

for name, styles, aug in [("自分のチームだけ", (0,), False),
                          ("自分＋知り合い3チーム", (0, 1, 2, 3), False),
                          ("自分＋知り合い3チーム＋拡張", (0, 1, 2, 3), True),
                          ("8チーム", tuple(range(8)), False)]:
    r = train_eval(styles, aug)
    print(f"{name:16s} ロボット 見つけた {r['robot']['recall']:.0%} 当たった {r['robot']['precision']:.0%}"
          f" ／ ボール 当たった {r['ball']['precision']:.0%}")

### 手を動かす 3：誤検出を減らすもう一つの工夫

In [ ]:
m = build_fomo()
m.compile(optimizer=keras.optimizers.Adam(2e-3), loss=fomo_loss())
X, L, _ = make_images(6000, seed=11, robot_styles=tuple(range(8)))
keras.utils.set_random_seed(0)
m.fit(X, grid_targets(L), validation_data=(X_val, grid_targets(L_val)), epochs=25, batch_size=64, verbose=0,
      callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
P = m.predict(X_unseen, verbose=0)
for th in [0.3, 0.5, 0.7, 0.9]:
    r = evaluate([fomo_detections(p, th) for p in P], L_unseen)["robot"]
    print(f"しきい値 {th}: 見つけた {r['recall']:.0%}  当たった {r['precision']:.0%}")

## 6-9 IR とカメラを組み合わせる

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part6-camera/fusion/

### 手を動かす 1：IR の推定を用意する

In [ ]:
import numpy as np
import keras
from rcjml import load_config
from rcjml.data import load_synthetic, split_by_session
from rcjml.nn import features, targets, keras_loss, build_ballnet, decode
from rcjml.sim import angle_error_deg
from rcjml.sim.camera import render, angle_from_x, FOV_DEG
from rcjml.vision import fomo_detections

cfg = load_config()
df = load_synthetic("ring12_large_normal")
train, test = split_by_session(df)
last = train.session.max()
fit, val = train[train.session < last], train[train.session == last]

keras.utils.set_random_seed(0)
ballnet = build_ballnet()
ballnet.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras_loss())
ballnet.fit(features(fit, cfg), targets(fit), validation_data=(features(val, cfg), targets(val)),
            epochs=400, batch_size=64, verbose=0,
            callbacks=[keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)])

ir = decode(ballnet.predict(features(test, cfg), verbose=0))["angle_deg"]
true = test.angle_deg.values

### 手を動かす 2：同じ場面のカメラ画像を作る

In [ ]:
rng = np.random.default_rng(0)
in_view = (np.abs(true) < FOV_DEG / 2) & (test.has_ball.values == 0)
idx = np.where(in_view)[0]
imgs = np.stack([render(rng, ball_at=(true[i], test.distance_cm.values[i]))[0] for i in idx])
P = fomo.predict(imgs, verbose=0)          # fomo は 6-6 で学習したもの

cam = np.full(len(test), np.nan)           # 見つからなければ NaN
for k, i in enumerate(idx):
    balls = [d for d in fomo_detections(P[k], 0.5) if d.cls == 0]
    if balls:
        guesses = [angle_from_x(d.cx) for d in balls]
        # ボールが 2 つ見えたら（反射など）、IR の角度に近いほうを選ぶ
        cam[i] = min(guesses, key=lambda a: abs((a - ir[i] + 180) % 360 - 180))

seen = ~np.isnan(cam)
print(f"前にある場面 {in_view.sum()} 個のうち、カメラで見つけた {seen.sum() / in_view.sum():.0%}")

### 手を動かす 3：混ぜる

In [ ]:
def fuse(ir_deg, cam_deg, s_ir=6.0, s_cam=1.5):
    out = ir_deg.copy()
    ok = ~np.isnan(cam_deg)
    w1, w2 = 1 / s_ir**2, 1 / s_cam**2
    a1, a2 = np.deg2rad(ir_deg[ok]), np.deg2rad(cam_deg[ok])
    out[ok] = np.rad2deg(np.arctan2(w1 * np.sin(a1) + w2 * np.sin(a2),
                                    w1 * np.cos(a1) + w2 * np.cos(a2)))
    return out

fused = fuse(ir, cam)
mae = lambda est, mask: angle_error_deg(est[mask], true[mask]).mean()
print(f"前にある場面: IR {mae(ir, in_view):.1f}°  カメラ {mae(cam, seen):.1f}°  混ぜた {mae(fused, in_view):.1f}°")
everywhere = np.ones(len(test), bool)
print(f"全部の場面:   IR {mae(ir, everywhere):.1f}°  混ぜた {mae(fused, everywhere):.1f}°")